# 模型選擇與架構設計

## 學習目標

完成本 Notebook 後，你將能夠：

1. 根據任務類型選擇合適的機器學習模型。
2. 理解資料規模、模型複雜度與過擬合風險之間的關係。
3. 比較不同模型在準確率、解釋性與推論成本上的差異。
4. 使用交叉驗證進行基本模型選擇。
5. 從實務限制角度思考模型部署與架構設計。

本章會以輕量的 sklearn 範例示範分類模型選擇、偏差與變異權衡，以及簡單的模型評估流程。


In [ ]:
# ── 環境設定 ────────────────────────────────────
# 載入本章節所需的 Python 套件，並建立可重現的隨機種子設定。

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification, make_moons
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print('環境設定完成')
print('可用套件：numpy, pandas, matplotlib, sklearn')


## 核心概念說明

模型選擇不是只挑選準確率最高的模型，而是要同時考量任務、資料、解釋性、資源限制與部署情境。

### 1. 任務類型與模型配對

- 分類任務：預測離散類別，例如客戶是否流失、郵件是否為垃圾郵件。
- 迴歸任務：預測連續數值，例如房價、銷售量、能源消耗。
- 非監督學習：從未標註資料中找出結構，例如分群或降維。
- 序列與時間序列任務：根據時間或順序資料預測未來狀態。

### 2. 資料規模與模型複雜度

- 小資料集通常適合簡單模型，例如邏輯迴歸、淺層決策樹。
- 中型資料集可嘗試隨機森林、梯度提升等較高表現力模型。
- 大型或非結構化資料可考慮深度學習，但需要更多運算資源。

### 3. 解釋性與效能取捨

線性模型與淺層決策樹通常較容易解釋；隨機森林、SVM 或神經網路可能有較佳預測能力，但決策過程較不直觀。

### 4. 偏差與變異權衡

簡單模型可能欠擬合，複雜模型可能過擬合。模型選擇的重點是在訓練表現與泛化能力之間取得平衡。


In [ ]:
# ── 示範：分類任務的模型比較 ────────────────────────────
# 這段程式碼建立一個分類資料集，並比較邏輯迴歸、決策樹、隨機森林與 SVM 的測試準確率。

import numpy as np
import pandas as pd

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

RANDOM_STATE = 42

X, y = make_classification(
    n_samples=1200,
    n_features=12,
    n_informative=6,
    n_redundant=2,
    n_classes=2,
    class_sep=1.2,
    random_state=RANDOM_STATE
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE
)

models = {
    '邏輯迴歸': make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    '決策樹': DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE),
    '隨機森林': RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE),
    'SVM': make_pipeline(StandardScaler(), SVC(kernel='rbf', C=1.0))
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    results.append({'模型': name, '測試準確率': round(acc, 4)})

result_df = pd.DataFrame(results).sort_values('測試準確率', ascending=False)
print(result_df.to_string(index=False))


## 如何解讀模型比較結果

若某個模型在測試集準確率較高，代表它在目前資料切分下有較好的泛化表現，但仍不能只根據單次切分做最終決策。

實務上可以進一步檢查：

- 模型是否穩定：不同資料切分下表現是否一致。
- 模型是否可解釋：業務或法規單位是否能理解模型決策。
- 推論是否夠快：是否符合即時服務需求。
- 訓練成本是否可接受：是否需要大量記憶體、GPU 或長時間訓練。
- 維運是否容易：模型上線後是否容易監控、重訓與除錯。

因此，最佳模型不一定是最複雜的模型，而是最符合任務需求與限制條件的模型。


## 偏差與變異的權衡

模型太簡單會欠擬合（偏差高），太複雜會過擬合（變異高）。以下用不同深度的決策樹，觀察訓練準確率與測試準確率如何隨模型複雜度變化，找出兩者開始拉開的轉折點。


In [ ]:
# ── 示範：偏差與變異權衡 ──────────────────────────────
# 這段程式碼使用非線性資料比較不同深度的決策樹，觀察模型複雜度對訓練與測試準確率的影響。

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

RANDOM_STATE = 42

X, y = make_moons(n_samples=800, noise=0.28, random_state=RANDOM_STATE)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=RANDOM_STATE
)

depths = [1, 2, 3, 5, 10, None]
records = []

for depth in depths:
    model = DecisionTreeClassifier(max_depth=depth, random_state=RANDOM_STATE)
    model.fit(X_train, y_train)
    train_acc = accuracy_score(y_train, model.predict(X_train))
    test_acc = accuracy_score(y_test, model.predict(X_test))
    records.append({
        'max_depth': '無限制' if depth is None else depth,
        '訓練準確率': train_acc,
        '測試準確率': test_acc
    })

score_df = pd.DataFrame(records)
print(score_df.round(4).to_string(index=False))

plt.figure(figsize=(8, 4))
plt.plot(score_df['max_depth'].astype(str), score_df['訓練準確率'], marker='o', label='訓練準確率')
plt.plot(score_df['max_depth'].astype(str), score_df['測試準確率'], marker='o', label='測試準確率')
plt.xlabel('決策樹深度')
plt.ylabel('Accuracy')
plt.title('模型複雜度與泛化表現')
plt.ylim(0.7, 1.05)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## 用交叉驗證比較候選模型

單次訓練測試切分的結果容易受切分方式影響。以下用交叉驗證比較邏輯迴歸、決策樹、隨機森林與 SVM 的平均表現與變異程度，再回頭對照上面提到的解釋性、推論速度與維運成本。


In [ ]:
# ── 實際應用：用交叉驗證選擇模型 ──────────────────────────
# 這段程式碼使用交叉驗證比較模型平均表現，避免只依賴單一次訓練測試切分。

import numpy as np
import pandas as pd

from sklearn.datasets import make_classification
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

RANDOM_STATE = 42

X, y = make_classification(
    n_samples=1000,
    n_features=10,
    n_informative=5,
    n_redundant=2,
    random_state=RANDOM_STATE
)

candidate_models = {
    '高解釋性：邏輯迴歸': make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    '簡單規則：決策樹': DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE),
    '高表現力：隨機森林': RandomForestClassifier(n_estimators=150, random_state=RANDOM_STATE),
    '非線性邊界：SVM': make_pipeline(StandardScaler(), SVC(kernel='rbf', C=1.0))
}

cv_results = []

for name, model in candidate_models.items():
    scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
    cv_results.append({
        '模型': name,
        '平均準確率': scores.mean(),
        '標準差': scores.std()
    })

cv_df = pd.DataFrame(cv_results).sort_values('平均準確率', ascending=False)
print(cv_df.round(4).to_string(index=False))

best_model = cv_df.iloc[0]
print(f"\n建議優先評估模型：{best_model['模型']}")
print(f"交叉驗證平均準確率：{best_model['平均準確率']:.4f} ± {best_model['標準差']:.4f}")
